# 06 ファクター・バックテスト(walk-forward + コスト + ベースライン比較)

`irp.backtest` でリゴラスに評価する:**weights は lag**(同足の先読みなし)、コストは**ターンオーバー**に課金、**walk-forward は purge+embargo でリーク無し**、そして**必ず単純ベースラインと並べる**(勝者だけでなく失敗も提示)。合成データなので数値は説明用で、結果そのものではない。

In [ ]:
from irp.utils.config import load_config
from irp import signals as S, backtest as B

cfg = load_config('backtest_config')
ann = cfg['evaluation']['annualization']
cost = B.CostModel.from_config(cfg)
print(f'annualization={ann}, cost per unit turnover={cost.per_unit:.4%} '
      f'(cost_bps={cost.cost_bps}, slippage_bps={cost.slippage_bps})')

In [ ]:
import numpy as np
import pandas as pd

# Synthetic panel with a MILD persistent cross-sectional structure (some names
# have higher risk-adjusted drift) so momentum has a small, honest edge to
# measure. No network. Real data: irp.data.get_prices / price_panel.
rng = np.random.default_rng(11)
idx = pd.bdate_range('2015-01-01', periods=1500)
names = [f'A{i}' for i in range(10)]
edge = np.linspace(-0.0003, 0.0004, len(names))  # persistent drift spread
vol = 0.012
rets = pd.DataFrame(
    {n: rng.normal(edge[i], vol, len(idx)) for i, n in enumerate(names)}, index=idx
)
prices = 100 * np.exp(rets.cumsum())
prices.iloc[-1].round(1)

## 信号 → weights → 月次リバランス → バックテスト

横断モメンタムを共通スキーマの信号にし、上位/下位 quantile の dollar-neutral weight に落とす。**lag してから**使い、月次リバランス(`rebalanced`)でターンオーバーを抑える。

In [ ]:
sig = S.momentum_signal(prices, lookback=252, skip=21)
raw_w = S.long_short_quantile(sig.lag(1).score, quantile=0.3)  # lag = no same-bar peek
w = B.rebalanced(raw_w, 'ME')                                  # hold monthly
res = B.run_backtest(w, rets, cost_model=cost, lag=1)
B.summary(res, periods=ann).round(4)

## ベースライン/ベンチと並べる

戦略は単独では意味を持たない。等加重バイ&ホールド(ベンチ代理)と、コストを切った同じ戦略(コストドラッグの可視化)を並べる。

In [ ]:
equal_weight = B.buy_and_hold(rets)                       # benchmark proxy
res_nocost = B.run_backtest(w, rets, cost_model=B.CostModel(0, 0), lag=1)
table = B.compare(
    {'momentum_LS': res, 'momentum_LS_nocost': res_nocost, 'equal_weight_BH': equal_weight},
    periods=ann,
)
table.round(4)

コスト有無の差が **ターンオーバーのドラッグ**。等加重との比較で、信号のスプレッドがコスト後も残るかを見る(合成データなので大小は構造次第)。

## walk-forward(out-of-sample)評価:リークしない fold

信号は causal なので全期間で計算しても先読みは無いが、評価は **test 窓のみ**(初期学習期間より後)で行うのが OOS。fold は purge+embargo でリーク無しを保証する。

In [ ]:
folds = B.walk_forward(prices.index, train=252, test=63, horizon=21, embargo=5)
leak_free = all(B.is_leakage_free(f, prices.index, horizon=21) for f in folds)
oos = prices.index[prices.index.isin(np.concatenate([f.test.values for f in folds]))]
print(f'{len(folds)} folds, all leakage-free: {leak_free}, OOS days: {len(oos)}')
print('first fold:', folds[0])

oos_res = B.run_backtest(w.loc[oos], rets.loc[oos], cost_model=cost, lag=1)
B.compare({'in_sample(full)': res, 'out_of_sample(test windows)': oos_res}, periods=ann).round(4)

in-sample と OOS の Sharpe 差は **過学習/期間依存** の温度計。(Phase 2b-2 でモデルを fold ごとに学習し、この OOS 窓で評価する。)

## コスト感応度(ターンオーバーの効き)

In [ ]:
rows = []
for bps in [0, 5, 10, 20, 50]:
    r = B.run_backtest(w, rets, cost_model=B.CostModel(bps, 0), lag=1)
    rows.append({'cost_bps': bps, 'ann_return': B.annualized_return(r, ann),
                 'sharpe': B.sharpe(r, ann), 'ann_turnover': B.annual_turnover(r, ann)})
pd.DataFrame(rows).set_index('cost_bps').round(4)

**正直な注記**: 合成データ・サバイバーシップ無視・単純コストモデル・税/借株料/資金制約は未考慮。これは *方法論* の実演であり投資結果ではない。次(Phase 2b-2)はモデル(Tier0-4)を walk-forward で学習し、同じエンジン/ベースライン/コストで評価する。